# Exploratory Data Analysis: Reddit Resume Dataset

This notebook explores the final cleaned datasets in `data/final/posts.parquet` (3,037 posts) and `data/final/comments.parquet` (8,101 comments), and generates publication-ready figures in `figures/` for the repository README.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

posts = pd.read_parquet("data/final/posts.parquet")
comments = pd.read_parquet("data/final/comments.parquet")
soc = pd.read_csv("data/reference/amended_soc_structure.csv")
soc_names = (
    soc.dropna(subset=["Detailed Occupation"])
    .set_index("Detailed Occupation")["Name"]
    .to_dict()
)

posts["created_dt"] = pd.to_datetime(posts["created_utc"], unit="s")
comments["created_dt"] = pd.to_datetime(comments["created_utc"], unit="s")

print(f"Posts: {posts.shape[0]:,} rows x {posts.shape[1]} cols")
print(f"Comments: {comments.shape[0]:,} rows x {comments.shape[1]} cols")
print(f"Date range: {posts['created_dt'].min().date()} to {posts['created_dt'].max().date()}")

Posts: 3,037 rows x 13 cols
Comments: 8,101 rows x 9 cols
Date range: 2026-01-01 to 2026-10-31


## 1. Temporal Coverage (Monthly Post Volume)

In [2]:
monthly_counts = posts.set_index("created_dt").resample("ME").size()
labels = [d.strftime("%b %Y") for d in monthly_counts.index]

fig, ax = plt.subplots(figsize=(8.5, 4.0))
bars = ax.bar(labels, monthly_counts.values, color="#2b5c8f", edgecolor="white", linewidth=0.6)
ax.set_title("Monthly Volume of Cleaned Resume Submissions (r/resumes, 2026)", fontweight="bold", pad=10)
ax.set_ylabel("Number of Posts")
ax.tick_params(axis="x", rotation=30)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 6, f"{int(h)}", ha="center", va="bottom", fontsize=8.5)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "monthly_post_volume.png", bbox_inches="tight")
plt.close(fig)

## 2. Years of Experience (YoE) Distribution

In [3]:
yoe_clipped = posts["yoe"].clip(upper=15)
median_yoe = posts["yoe"].median()
mean_yoe = posts["yoe"].mean()

fig, ax = plt.subplots(figsize=(8.5, 4.0))
ax.hist(yoe_clipped, bins=16, range=(-0.5, 15.5), color="#3a7ca5", edgecolor="white", linewidth=0.8)
ax.axvline(median_yoe, color="#d95f02", linestyle="--", linewidth=1.8, label=f"Median: {median_yoe:.1f} YoE")
ax.axvline(mean_yoe, color="#1b9e77", linestyle=":", linewidth=1.8, label=f"Mean: {mean_yoe:.2f} YoE")
ax.set_xticks(range(0, 16))
ax.set_xticklabels([str(i) if i < 15 else "15+" for i in range(16)])
ax.set_title("Distribution of Applicant Years of Experience (Clipped at 15+ YoE)", fontweight="bold", pad=10)
ax.set_xlabel("Years of Experience")
ax.set_ylabel("Number of Posts")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "yoe_distribution.png", bbox_inches="tight")
plt.close(fig)

print(posts["yoe"].describe())

count    3037.000000
mean        2.718900
std         3.614991
min         0.000000
25%         0.000000
50%         2.000000
75%         4.000000
max        45.000000
Name: yoe, dtype: float64


## 3. Geographic & Work-Mode Distribution

In [4]:
top_countries = posts["country"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.barh(top_countries.index, top_countries.values, color="#4a7c59", edgecolor="white")
ax.set_title("Top 10 Normalized Locations & Work Modes", fontweight="bold", pad=10)
ax.set_xlabel("Number of Posts")
for bar in bars:
    w = bar.get_width()
    pct = 100 * w / len(posts)
    ax.text(w + 20, bar.get_y() + bar.get_height() / 2, f"{int(w):,} ({pct:.1f}%)", va="center", fontsize=8.5)
ax.set_xlim(0, top_countries.max() * 1.22)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "top_countries.png", bbox_inches="tight")
plt.close(fig)

## 4. Current Roles vs. Target Roles (Amended 2018 SOC)

In [5]:
def short_label(code: str, max_len: int = 34) -> str:
    name = soc_names.get(code, code).split(",")[0].strip()
    if len(name) > max_len:
        name = name[: max_len - 1] + "…"
    return f"{name} ({code})"

top_curr = posts["current_role"].value_counts().head(8).sort_values()
top_targ = posts["target_role"].value_counts().head(8).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

bars1 = axes[0].barh([short_label(c) for c in top_curr.index], top_curr.values, color="#6c757d")
axes[0].set_title("Top 8 Current Roles / Statuses", fontweight="bold")
axes[0].set_xlabel("Number of Posts")
axes[0].set_xlim(0, top_curr.max() * 1.18)
for b in bars1:
    axes[0].text(b.get_width() + 10, b.get_y() + b.get_height() / 2, f"{int(b.get_width())}", va="center", fontsize=8.5)

bars2 = axes[1].barh([short_label(c) for c in top_targ.index], top_targ.values, color="#2b5c8f")
axes[1].set_title("Top 8 Target Roles", fontweight="bold")
axes[1].set_xlabel("Number of Posts")
axes[1].set_xlim(0, top_targ.max() * 1.18)
for b in bars2:
    axes[1].text(b.get_width() + 10, b.get_y() + b.get_height() / 2, f"{int(b.get_width())}", va="center", fontsize=8.5)

fig.suptitle("Standardized Current vs. Target Occupations (Amended 2018 SOC)", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "top_roles_comparison.png", bbox_inches="tight")
plt.close(fig)

## 5. Comment Relevance & Actionable Feedback Density

In [6]:
rel_counts = comments["is_relevant"].value_counts()
rel_per_post = comments[comments["is_relevant"]].groupby("post_id").size().clip(upper=10)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

labels_rel = ["Actionable Advice\n(True)", "Non-Advice / Other\n(False)"]
vals_rel = [int(rel_counts.get(True, 0)), int(rel_counts.get(False, 0))]
bars = axes[0].bar(labels_rel, vals_rel, color=["#1b9e77", "#adb5bd"], width=0.55)
axes[0].set_title("Comment Relevance Classification (n = 8,101)", fontweight="bold")
axes[0].set_ylabel("Number of Comments")
axes[0].set_ylim(0, max(vals_rel) * 1.15)
for b in bars:
    h = b.get_height()
    axes[0].text(b.get_x() + b.get_width() / 2, h + 60, f"{h:,} ({100*h/len(comments):.1f}%)", ha="center", fontsize=9)

axes[1].hist(rel_per_post, bins=10, range=(0.5, 10.5), color="#1b9e77", edgecolor="white", linewidth=0.8)
axes[1].set_xticks(range(1, 11))
axes[1].set_xticklabels([str(i) if i < 10 else "10+" for i in range(1, 11)])
axes[1].set_title("Actionable Comments per Post (n = 1,678 Posts)", fontweight="bold")
axes[1].set_xlabel("Actionable Comments")
axes[1].set_ylabel("Number of Posts")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "comment_relevance_distribution.png", bbox_inches="tight")
plt.close(fig)